In [1]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

# Load the dataset
with open("counties.json", "r") as f:
    records = json.load(f)

# Flatten nested objects into columns like noaa.temp, race.hispanic_male, etc.
df = pd.json_normalize(records, sep=".")

# For each column, change spaces to underscores and convert to lowercase
df.columns = df.columns.str.replace(" ", "_").str.lower()

df.head()

,name,fips,state,land_area_(km^2),area_(km^2),longitude_(deg),latitude_(deg),zip-codes,male,female,...,"industry.agriculture,_forestry,_fishing_and_hunting.payroll","industry.agriculture,_forestry,_fishing_and_hunting.employees",industry.utilities.payroll,industry.utilities.employees,industry.management_of_companies_and_enterprises.payroll,industry.management_of_companies_and_enterprises.employees,industry.industries_not_classified.payroll,industry.industries_not_classified.employees,"industry.mining,_quarrying,_and_oil_and_gas_extraction.payroll","industry.mining,_quarrying,_and_oil_and_gas_extraction.employees"
0,cuming county,31039,NE,1477.641638,1488.343176,-96.787366,41.916346,"[68047, 68641, 68004, 68045, 68788, 68716, 687...",4435,4411,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,lancaster county,31109,NE,2169.252486,2192.120047,-96.688211,40.784132,"[68503, 68428, 68526, 68301, 68419, 68017, 683...",160211,158879,...,730000.0,111.0,8730000.0,95.0,165117000.0,2024.0,150000.0,3.0,NaN,NaN
2,nuckolls county,31129,NE,1489.645186,1491.363670,-98.047277,40.176383,"[68957, 68943, 68325, 68935, 68974, 68964, 689...",2059,2089,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,keith county,31101,NE,2749.531887,2874.204062,-101.657059,41.198294,"[69121, 69155, 69146, 69153, 69165, 69147, 691...",4052,3982,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,phelps county,31137,NE,1398.048570,1399.695104,-99.414593,40.513105,"[68845, 68924, 68949, 68836, 68940, 68863, 689...",4527,4507,...,2557000.0,47.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
# Feature engineering

# Divide the number of covid deaths in each county in 2022-03-01 by the 2019 population – add a column called covid-deaths total per capita containing this data to the dataframe.
df["covid-deaths_total_per_capita"] = df["covid-deaths.2022-03-01"] / df["population.2019"]

# Divide the number of covid confirmed cases in each county in 2022-03-01 by the 2019 population – add a column called covid-confirmed total per capita containing this data to the dataframe.
df["covid-confirmed_total_per_capita"] = df["covid-confirmed.2022-03-01"] / df["population.2019"]

# Create an indicator that measures whether a county has greater-than-average life expectancy. Add a column called above average life-expectancy containing this data to the dataframe.
avg_life_expectancy = df["life-expectancy"].mean()
df["above_average_life_expectancy"] = df["life-expectancy"] > avg_life_expectancy

# Record for each county the largest industry by number of employees. You will need to use the 20 variables in the data named industry/.../employees. To do this, you will need to fill missing values in these variables. For simplicity, fill missing values with 0 (you may find .fillna useful). Put this information into a single column called biggest industry.
industry_cols = [col for col in df.columns if "industry" in col and "employees" in col]

for col in industry_cols:
    df[col] = df[col].fillna(0)

df["biggest_industry"] = df[industry_cols].idxmax(axis=1).str.split(".").str[1]

# Record for each county the modal educational level. You will need to use the 4 variables in the data named edu/.... Put this information into a single column called county_modal_ed.

edu_cols = [col for col in df.columns if "edu." in col] # We use `edu.` since `edu` is included in the name of some industry columns

for col in edu_cols:
    df[col] = df[col].fillna(0)

df["county_modal_ed"] = df[edu_cols].idxmax(axis=1).str.split(".").str[1]

print(df[["covid-deaths_total_per_capita", "covid-confirmed_total_per_capita","above_average_life_expectancy", "biggest_industry", "county_modal_ed"]].head())

   covid-deaths_total_per_capita  covid-confirmed_total_per_capita  \
0                       0.001696                          0.203934   
1                       0.000533                          0.243025   
2                       0.002652                          0.231678   
3                       0.001245                          0.190192   
4                       0.001107                          0.256254   

   above_average_life_expectancy                   biggest_industry  \
0                           True                      manufacturing   
1                           True  health_care_and_social_assistance   
2                           True  health_care_and_social_assistance   
3                           True                       retail_trade   
4                           True                      manufacturing   

  county_modal_ed  
0     high-school  
1      bachelors+  
2    some-college  
3    some-college  
4    some-college  


(a) Using the mean as the cutoff is reasonable for creating a simple, balanced binary indicator, but it is not always ideal. If the `life-expectancy` distribution is skewed or has outliers, the mean can be pulled away from the center of most counties. In that case, the median or quantile-based thresholds could be more robust.

(b) A hypothesis-test-based approach would be to compare each county's life expectancy to the overall population average using a one-sample test (conceptually) and classify counties as significantly above average, significantly below average, or not significantly different. That would produce 3 categories instead of 2.

(c) Filling missing industry employment values with 0 assumes missing means no employees (or effectively no activity) in that industry. This can be wrong if data are missing for reporting/collection reasons, and it may bias the "biggest_industry" feature by making some industries look artificially small.

In [3]:
X = ['state'
    ,'longitude_(deg)'
    ,'latitude_(deg)'
    ,'noaa.temp'
    ,'noaa.altitude'
    ,'male'
    ,'deaths.suicides'
    ,'deaths.homicides'
    ,'bls.2020.unemployed'
    ,'avg_income'
    ,'covid-deaths_total_per_capita'    #constructed
    ,'covid-confirmed_total_per_capita'    #constructed
    ,'covid-vaccination.2021-12-01'
    ,'county_modal_ed'    #constructed
    ,'poverty-rate'
    ,'cost-of-living.living_wage'
    ,'cost-of-living.food_costs'
    ,'cost-of-living.medical_costs'
    ,'cost-of-living.housing_costs'
    ,'cost-of-living.tax_costs'
    ,'health.average_number_of_mentally_unhealthy_days'
    ,'health.%_smokers'
    ,'health.%_adults_with_obesity'
    ,'health.%_physically_inactive'
    ,'health.%_long_commute_-_drives_alone'
    ,'biggest_industry']    #constructed

samples = []

for x in X:
    if df[x].dtype == "int64" or df[x].dtype == "float64":
        numeric_data = df[["life-expectancy", x]].dropna()
        test_result = scipy.stats.linregress(numeric_data["life-expectancy"], numeric_data[x])
        samples.append((x, test_result))
    else:
        categorical_data = df[["life-expectancy", x]].dropna()
        samples_by_group = []
        for value in set(categorical_data[x]):
            mask = categorical_data[x] == value
            samples_by_group.append(categorical_data['life-expectancy'][mask])

        stat, p = scipy.stats.kruskal(*samples_by_group)
        samples.append((x, (stat, p)))

alpha = 0.05
summary_rows = []

for variable, result in samples:
    if hasattr(result, "pvalue"):
        test_statistic = result.rvalue
        p_value = result.pvalue
    else:
        test_statistic, p_value = result

    summary_rows.append({
        "variable": variable,
        "test_statistic": test_statistic,
        "p_value": p_value,
        "significant_at_0.05": p_value < alpha
    })

results_table = pd.DataFrame(summary_rows)
results_table["test_statistic"] = results_table["test_statistic"].astype(float)
results_table["p_value"] = results_table["p_value"].astype(float)

results_table

,variable,test_statistic,p_value,significant_at_0.05
0,state,1719.974217,0.000000e+00,True
1,longitude_(deg),-0.185017,1.358906e-25,True
2,latitude_(deg),0.437397,5.304361e-147,True
3,noaa.temp,-0.472559,1.463919e-174,True
4,noaa.altitude,0.268992,3.235692e-53,True
5,male,0.170727,5.584895e-22,True
6,deaths.suicides,0.170174,9.216754e-21,True
7,deaths.homicides,0.088597,3.552415e-05,True
8,bls.2020.unemployed,0.142672,9.946132e-16,True
9,avg_income,0.564267,1.111643e-263,True


(a) Rejecting the null hypothesis in a Pearson correlation test means there is evidence that the true linear correlation is not zero. In practice, the two numeric variables have a statistically detectable linear association.

(b) Pearson's correlation coefficient `r` measures direction and strength of linear relationship. Positive `r` means both variables tend to increase together, negative `r` means one increases as the other decreases, and values closer to `|1|` indicate stronger linear relationships. This helps identify potentially predictive variables and expected trend direction.

(c) In your results, very strong significance for variables like `health.%_smokers`, `health.%_physically_inactive`, `poverty-rate`, and `health.average_number_of_mentally_unhealthy_days` aligns with common expectations because these factors are plausibly linked to health outcomes. A variable like `state` also showed very strong significance, which can happen due to broad regional policy, demographic, environmental, and healthcare-system differences.

(d) Pearson assumes approximate linearity, sensitivity to outliers, and for strict inference, normality/homoscedasticity assumptions in residual behavior. Spearman is rank-based and captures monotonic (not strictly linear) relationships, making it more robust to outliers and non-normality. You can verify assumptions with scatterplots, residual plots, QQ-plots, and by checking whether rank-based patterns are clearer than linear ones.

(e) Rejecting the null hypothesis in a Kruskal-Wallis test means at least one group's distribution (often interpreted as central tendency) differs from the others. It does not by itself identify which specific groups differ.

(f) Kruskal-Wallis is appropriate because it is a non-parametric alternative to one-way ANOVA and does not require normality of each group. It is useful when comparing more than two independent groups for an ordinal or continuous response under weaker distributional assumptions.

(g) With more than 20 hypothesis tests at `alpha = 0.05`, the chance of at least one type I error across the whole family of tests is higher than 5 percent. A rough independent-test approximation is `1 - (1 - 0.05)^20`, which is about 64 percent. This motivates multiple-comparison controls (for example Bonferroni or FDR procedures).

## Classification on Above Average Life Expectancy

In this section, we use `above_average_life_expectancy` as the dependent variable and test each predictor for association:
- Numerical predictors: Kruskal-Wallis test (two groups defined by `above_average_life_expectancy`).
- Categorical predictors: Chi-squared test of independence (`chi2_contingency`).

In [4]:
# Hypothesis tests with above_average_life_expectancy as the response

response = "above_average_life_expectancy"
alpha = 0.05
classification_rows = []

# Use the same predictor set as above
predictors = X.copy()

for var in predictors:
    if var == response:
        continue

    # Numeric predictor -> Kruskal-Wallis across the two response groups
    if pd.api.types.is_numeric_dtype(df[var]):
        subset = df[[var, response]].dropna()

        group_false = subset.loc[subset[response] == False, var]
        group_true = subset.loc[subset[response] == True, var]

        # Guard against degenerate cases with empty groups
        if len(group_false) > 0 and len(group_true) > 0:
            stat, p_value = scipy.stats.kruskal(group_false, group_true)
        else:
            stat, p_value = np.nan, np.nan

        classification_rows.append(
            {
                "variable": var,
                "test": "kruskal",
                "test_statistic": stat,
                "p_value": p_value,
                "significant_at_0.05": (p_value < alpha) if pd.notna(p_value) else False,
            }
        )

    # Categorical predictor -> Chi-squared test of independence
    else:
        subset = df[[var, response]].dropna()
        contingency = pd.crosstab(subset[var], subset[response])

        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2_stat, p_value, _, _ = scipy.stats.chi2_contingency(contingency)
        else:
            chi2_stat, p_value = np.nan, np.nan

        classification_rows.append(
            {
                "variable": var,
                "test": "chi2_independence",
                "test_statistic": chi2_stat,
                "p_value": p_value,
                "significant_at_0.05": (p_value < alpha) if pd.notna(p_value) else False,
            }
        )

classification_results_table = pd.DataFrame(classification_rows)
classification_results_table["test_statistic"] = pd.to_numeric(
    classification_results_table["test_statistic"], errors="coerce"
)
classification_results_table["p_value"] = pd.to_numeric(
    classification_results_table["p_value"], errors="coerce"
)
classification_results_table = classification_results_table.sort_values("p_value")

classification_results_table

,variable,test,test_statistic,p_value,significant_at_0.05
20,health.average_number_of_mentally_unhealthy_days,kruskal,1382.618831,1.257219e-302,True
0,state,chi2_independence,1422.920133,4.904640e-265,True
21,health.%_smokers,kruskal,1195.214696,6.687765e-262,True
14,poverty-rate,kruskal,1176.941922,6.258975e-258,True
9,avg_income,kruskal,1000.681592,1.276769e-219,True
16,cost-of-living.food_costs,kruskal,880.784199,1.476233e-193,True
2,latitude_(deg),kruskal,879.962514,2.227326e-193,True
23,health.%_physically_inactive,kruskal,876.676598,1.153777e-192,True
3,noaa.temp,kruskal,836.581209,6.009898e-184,True
10,covid-deaths_total_per_capita,kruskal,574.849159,4.948824e-127,True


In [5]:
# Compare significance against the previous section (continuous life-expectancy response)

previous_sig = set(results_table.loc[results_table["significant_at_0.05"], "variable"])
classification_sig = set(
    classification_results_table.loc[
        classification_results_table["significant_at_0.05"], "variable"
    ]
)

comparison = pd.DataFrame(
    {
        "significant_with_life_expectancy_numeric_response": sorted(previous_sig),
    }
)

print("Significant only with numeric response:", sorted(previous_sig - classification_sig))
print("Significant only with binary response:", sorted(classification_sig - previous_sig))
print("Significant in both:", sorted(previous_sig & classification_sig))

Significant only with numeric response: []
Significant only with binary response: []
Significant in both: ['avg_income', 'biggest_industry', 'bls.2020.unemployed', 'cost-of-living.food_costs', 'cost-of-living.housing_costs', 'cost-of-living.living_wage', 'cost-of-living.medical_costs', 'cost-of-living.tax_costs', 'county_modal_ed', 'covid-confirmed_total_per_capita', 'covid-deaths_total_per_capita', 'covid-vaccination.2021-12-01', 'deaths.homicides', 'deaths.suicides', 'health.%_adults_with_obesity', 'health.%_long_commute_-_drives_alone', 'health.%_physically_inactive', 'health.%_smokers', 'health.average_number_of_mentally_unhealthy_days', 'latitude_(deg)', 'longitude_(deg)', 'male', 'noaa.altitude', 'noaa.temp', 'poverty-rate', 'state']


(a) Rejecting the null hypothesis in a chi-squared test of independence means there is evidence that the two categorical variables are associated (not independent). In this section, it means the predictor and `above_average_life_expectancy` are unlikely to be independent.

(b) Hypothesis tests provide a principled way to separate likely signal from noise and to quantify uncertainty with p-values. However, they have limitations: p-values are sensitive to sample size, statistical significance is not the same as practical importance, assumptions can be violated, and running many tests inflates the risk of false positives unless multiple-testing corrections are considered.

(c) In this run, the significant predictor set was the same for both response versions (continuous `life-expectancy` and binary `above_average_life_expectancy`). So in this dataset, binarizing did not change which variables were flagged at `alpha = 0.05`. In general, this is not guaranteed: classification and regression can emphasize different signal, so feature selection should still be validated for the specific modeling task.

(d) A key challenge is choosing tests that match variable types and assumptions, then interpreting significance carefully rather than mechanically. Important lessons are to check data quality and missingness handling, report effect sizes or direction where possible, and account for multiple testing to reduce false discoveries.